# Sea Sentinel: Comprehensive Project Exploratory Data Analysis (EDA)
### Sea Sentinel — AI-Powered Underwater Debris & Anomaly Detection
---


## 1. Executive Summary & Problem Context
Side-Scan Sonar (SSS) acoustic surveys produce continuous waterfall backscatter rasters. Anthropogenic marine debris presents acoustic highlights followed by trailing acoustic shadows.
This notebook delivers a comprehensive Exploratory Data Analysis (EDA) of the Sea Sentinel dataset across acoustic intensity, class distributions, bounding box geometries, and semantic mask dimensions.

In [ ]:
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "eda" else Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "yolo"
print(f"Loading data from: {DATA_DIR}")

## 2. Dataset Size, Formats & Split Analysis
Quantifying image counts across train, validation, and test splits.

In [ ]:
train_imgs = list((DATA_DIR / "images" / "train").glob("*.*"))
val_imgs = list((DATA_DIR / "images" / "val").glob("*.*"))
test_imgs = list((DATA_DIR / "images" / "test").glob("*.*"))

split_df = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing", "Total"],
    "Image Count": [len(train_imgs), len(val_imgs), len(test_imgs), len(train_imgs) + len(val_imgs) + len(test_imgs)],
    "Percentage (%)": [
        round(100 * len(train_imgs) / max(1, len(train_imgs)+len(val_imgs)+len(test_imgs)), 1),
        round(100 * len(val_imgs) / max(1, len(train_imgs)+len(val_imgs)+len(test_imgs)), 1),
        round(100 * len(test_imgs) / max(1, len(train_imgs)+len(val_imgs)+len(test_imgs)), 1),
        100.0
    ]
})
display(split_df)

## 3. Acoustic Backscatter & Rayleigh Speckle Distribution
Acoustic intensity histograms and Michelson contrast distributions across the sonar dataset.

In [ ]:
sample_imgs = train_imgs[:50]
means, stds, contrasts = [], [], []

for p in sample_imgs:
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        means.append(np.mean(img))
        stds.append(np.std(img))
        contrast = (np.max(img) - np.min(img)) / max(1.0, float(np.max(img) + np.min(img)))
        contrasts.append(contrast)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=means, y=stds, color='navy', ax=ax[0])
ax[0].set_title("Mean vs. Standard Deviation (Speckle Variance)", fontweight="bold")
ax[0].set_xlabel("Mean Acoustic Backscatter")
ax[0].set_ylabel("Standard Deviation")

ax[1].hist(contrasts, bins=20, color='teal', edgecolor='black')
ax[1].set_title("Michelson Contrast Distribution", fontweight="bold")
ax[1].set_xlabel("Contrast Value")
plt.tight_layout()
plt.show()